In [ ]:
import os
from google.colab import drive

# 1. Installazione dipendenze di colab-only
!pip install -q kagglehub

# 2. Monta Drive
drive.mount('/content/drive')

# 3. Definisci i percorsi su Drive
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
INDEX_DIR = os.path.join(BASE_DRIVE, 'indexes')
PROCESSED_DIR = os.path.join(BASE_DRIVE, 'data/processed')

os.makedirs(INDEX_DIR, exist_ok=True)
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(PROCESSED_DIR, 'images', split), exist_ok=True)

# 4. Download del dataset tramite kagglehub (gestisce la cache locale di Colab)
import kagglehub
print("⬇️ Scaricamento dataset da Kaggle in corso...")
raw_data_path = kagglehub.dataset_download("timoboz/clevr-dataset")
print(f"✅ Dataset pronto nella cache locale: {raw_data_path}")

In [ ]:
import json
import random

print("🔍 Mappatura veloce dei file originali in memoria...")

# Dizionario per O(1) lookup: nome_file -> percorso_assoluto
raw_files_map = {}
for root, dirs, files in os.walk(raw_data_path):
    for file in files:
        raw_files_map[file] = os.path.join(root, file)

raw_train_json = raw_files_map.get('CLEVR_train_questions.json')
raw_val_json = raw_files_map.get('CLEVR_val_questions.json')

if not raw_train_json or not raw_val_json:
    raise FileNotFoundError("File JSON originali non trovati nella cache!")
    
print("✅ Mappatura completata.")

In [ ]:
print("⚙️ Generazione e salvataggio degli indici...")

# Caricamento JSON
with open(raw_train_json, 'r') as f:
    train_full = json.load(f)['questions']
with open(raw_val_json, 'r') as f:
    val_full = json.load(f)['questions']

# SPLIT DETERMINISTICO (Fondamentale per riproducibilità)
random.seed(42)
random.shuffle(train_full)
random.shuffle(val_full)

# Configurazione campioni: 
# Test Set estratto dal Val Set per possedere la ground truth dei ragionamenti
splits = {
    'train': train_full[:15000],
    'val': val_full[:1000],
    'test': val_full[1000:2000]
}

# Salvataggio degli Indici in Drive
for name, data in splits.items():
    index_path = os.path.join(INDEX_DIR, f'{name}_index.json')
    with open(index_path, 'w') as f:
        json.dump({'questions': data}, f)

print(f"✅ Indici creati! Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

In [ ]:
import shutil
from tqdm import tqdm

print("🚀 Avvio trasferimento immagini su Drive (Modalità Idempotente e Veloce)...")

for split_name, subset in splits.items():
    target_dir = os.path.join(PROCESSED_DIR, 'images', split_name)
    
    # 1. Identifichiamo le immagini UNICHE necessarie per questo split
    unique_images = set([item['image_filename'] for item in subset])
    
    # 2. Filtriamo solo quelle che non sono ancora su Drive o che sono corrotte (0 byte)
    missing_images = [
        img for img in unique_images 
        if not os.path.exists(os.path.join(target_dir, img)) or os.path.getsize(os.path.join(target_dir, img)) == 0
    ]
    
    if not missing_images:
        print(f"✅ {split_name.upper()}: Tutte le {len(unique_images)} immagini sono già presenti e intatte.")
        continue
        
    # 3. Copia chirurgica
    print(f"⚠️ {split_name.upper()}: Copia di {len(missing_images)} immagini mancanti in corso...")
    for img_name in tqdm(missing_images, desc=f"Copia {split_name}"):
        src_path = raw_files_map.get(img_name)
        dst_path = os.path.join(target_dir, img_name)
        
        if src_path:
            shutil.copy2(src_path, dst_path) # copy2 preserva i metadati
        else:
            print(f"\n❌ ERRORE: {img_name} non trovata nel dataset originale!")

print("\n🏁 Ecosistema Dati pronto per l'addestramento!")